# PySpark reference 8: window functions

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-reference-08-window-functions.ipynb)

Window specifications, every window function with its parameters, aggregates over windows, ranking and top-N, deduplication, `lag`/`lead`, frames, running and moving metrics, forward fill, sessions, gaps-and-islands, SQL syntax and window performance. Practise it in [PySpark practice 8: window functions](pyspark-practice-08-window-functions.ipynb).

*Last reviewed: November 2026, against Apache Spark 4.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

A local session, the shared `customers`, `orders`, `order_items`, `products` and `events` tables (customer 4 has no region, customers 7 and 8 never ordered, `SKU-4014` has no product row, one event is an exact retry), and small helpers: `plan_lines(df)` returns what `explain()` prints, `final_plan(df)` runs a query and returns the plan Spark actually used, and `expected_error()` prints the error class of an example that fails on purpose. `by_time` is the ordered per-customer window used by the function catalog.

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import Window
from pyspark.sql import functions as F

customers = spark.createDataFrame([
    (1, "Acme Corp", "North", date(2024, 1, 3)),
    (2, "Bright Retail", "North", date(2024, 2, 11)),
    (3, "Cyan Traders", "South", date(2024, 3, 15)),
    (4, "Delta Grocers", None, date(2024, 5, 2)),  # region never arrived from the CRM
    (5, "Echo Supplies", "West", date(2024, 6, 20)),
    (6, "Foxtrot Foods", "West", date(2024, 8, 1)),
    (7, "Gamma Sports", "East", date(2024, 9, 14)),  # 7 and 8 have never ordered
    (8, "Hotel Homeware", "East", date(2024, 11, 5)),
], "customer_id int, name string, region string, signup_date date")

orders = spark.createDataFrame([
    (1, 1, "paid", datetime(2026, 1, 5, 9, 15), D("120.00")),
    (2, 1, "shipped", datetime(2026, 1, 5, 10, 5), D("84.50")),
    (3, 1, "returned", datetime(2026, 1, 5, 11, 40), D("30.00")),
    (4, 2, "paid", datetime(2026, 1, 6, 15, 20), D("260.00")),
    (5, 2, "placed", datetime(2026, 1, 7, 8, 5), D("55.25")),
    (6, 3, "shipped", datetime(2026, 1, 7, 17, 45), D("149.00")),
    (7, 3, "paid", datetime(2026, 1, 9, 12, 10), D("310.00")),
    (8, 4, "returned", datetime(2026, 1, 10, 19, 30), D("89.90")),
    (9, 4, "shipped", datetime(2026, 1, 12, 8, 55), D("1180.00")),
    (10, 5, "paid", datetime(2026, 1, 12, 13, 0), D("74.00")),
    (11, 6, "placed", datetime(2026, 1, 13, 10, 25), D("210.00")),
    (12, 6, "shipped", datetime(2026, 1, 14, 0, 30), D("96.40")),  # just after midnight UTC
], "order_id int, customer_id int, status string, order_ts timestamp, amount decimal(12,2)")

order_items = spark.createDataFrame([
    (1, "SKU-4001", 2, D("12.50")), (1, "SKU-4007", 1, D("95.00")),
    (2, "SKU-4002", 1, D("34.50")), (2, "SKU-4009", 2, D("25.00")),
    (3, "SKU-4002", 1, D("30.00")),
    (4, "SKU-4003", 1, D("149.00")), (4, "SKU-4004", 2, D("55.50")),
    (5, "SKU-4005", 1, D("55.25")),
    (6, "SKU-4003", 1, D("149.00")),
    (7, "SKU-4006", 1, D("189.00")), (7, "SKU-4008", 1, D("121.00")),
    (8, "SKU-4010", 1, D("89.90")),
    (9, "SKU-4011", 20, D("59.00")),
    (10, "SKU-4012", 2, D("37.00")),
    (11, "SKU-4013", 1, D("210.00")),
    (12, "SKU-4014", 1, D("41.40")), (12, "SKU-4005", 1, D("55.00")),
], "order_id int, sku string, qty int, unit_price decimal(8,2)")

products = spark.createDataFrame([
    ("SKU-4001", "Kitchen", D("12.50")), ("SKU-4002", "Kitchen", D("34.50")),
    ("SKU-4003", "Garden", D("149.00")), ("SKU-4004", "Garden", D("55.50")),
    ("SKU-4005", "Pantry", D("55.25")), ("SKU-4006", "Outdoor", D("189.00")),
    ("SKU-4007", "Kitchen", D("95.00")), ("SKU-4008", "Outdoor", D("121.00")),
    ("SKU-4009", "Pantry", D("25.00")), ("SKU-4010", "Garden", D("89.90")),
    ("SKU-4011", "Pantry", D("59.00")), ("SKU-4012", "Kitchen", D("37.00")),
    ("SKU-4013", "Outdoor", D("210.00")),  # SKU-4014 is missing
], "sku string, category string, list_price decimal(8,2)")

events = spark.createDataFrame([
    (1, "view", datetime(2026, 1, 6, 9, 0)), (1, "add_to_cart", datetime(2026, 1, 6, 9, 2)),
    (1, "purchase", datetime(2026, 1, 6, 9, 10)), (2, "view", datetime(2026, 1, 7, 10, 0)),
    (2, "purchase", datetime(2026, 1, 7, 10, 40)), (3, "view", datetime(2026, 1, 6, 11, 15)),
    (3, "view", datetime(2026, 1, 8, 11, 20)), (4, "add_to_cart", datetime(2026, 1, 9, 20, 5)),
    (4, "purchase", datetime(2026, 1, 10, 8, 30)), (5, "view", datetime(2026, 1, 11, 14, 45)),
    (5, "view", datetime(2026, 1, 12, 9, 0)), (5, "purchase", datetime(2026, 1, 12, 12, 30)),
    (5, "view", datetime(2026, 1, 12, 9, 0)),  # an SDK retry committed this event twice
], "user_id int, event string, ts timestamp")

import contextlib
import io
import logging
import re


def plan_lines(df, mode="simple"):
    """The lines df.explain(mode) prints."""
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        df.explain(mode)
    return buffer.getvalue().splitlines()


def final_plan(df):
    """Run df, then return the plan Spark actually used (AQE's final plan when there is one)."""
    df.collect()
    text = "\n".join(plan_lines(df))
    if "== Final Plan ==" in text:
        text = text.split("== Final Plan ==")[1].split("== Initial Plan ==")[0]
    return text


@contextlib.contextmanager
def expected_error():
    """Run a block that is meant to fail and print the error class instead of a stack trace."""
    spark.sparkContext.setLogLevel("OFF")
    for logger in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
        logging.getLogger(logger).setLevel(logging.CRITICAL)
    try:
        yield
    except Exception as error:
        print(type(error).__name__ + ":", str(error).split(".")[0][:100])
    finally:
        spark.sparkContext.setLogLevel("ERROR")

by_time = Window.partitionBy("customer_id").orderBy("order_ts", "order_id")  # used by the function catalog

## Contents

| section | entries |
| --- | --- |
| [Window specifications](#window-specifications) | [`Window` and `WindowSpec`](#window-and-windowspec), [How a window works](#how-a-window-works) |
| [Window function catalog](#window-function-catalog) | [`row_number`](#row_number), [`rank`, `dense_rank`](#rank-dense_rank), [`percent_rank`, `cume_dist`](#percent_rank-cume_dist), [`ntile`](#ntile), [`lag`, `lead`](#lag-lead), [`nth_value`](#nth_value), [`first`, `last` over a window](#first-last-over-a-window), [Aggregates over a window](#aggregates-over-a-window) |
| [Aggregates and ranking](#aggregates-and-ranking) | [Aggregates over a whole partition](#aggregates-over-a-whole-partition), [Rank rows: row_number, rank and dense_rank](#rank-rows-row_number-rank-and-dense_rank), [Relative position: percent_rank, cume_dist and ntile](#relative-position-percent_rank-cume_dist-and-ntile), [Keep the top N rows per group](#keep-the-top-n-rows-per-group) |
| [Deduplication and offsets](#deduplication-and-offsets) | [Deduplicate: keep the latest record per key](#deduplicate-keep-the-latest-record-per-key), [Look back and ahead: lag and lead](#look-back-and-ahead-lag-and-lead), [Detect changes and number the runs](#detect-changes-and-number-the-runs), [first, last and nth_value](#first-last-and-nth_value) |
| [Frames and running metrics](#frames-and-running-metrics) | [Frames: choose which rows each row sees](#frames-choose-which-rows-each-row-sees), [Running totals and cumulative metrics](#running-totals-and-cumulative-metrics), [Moving windows: last N rows vs last N days](#moving-windows-last-n-rows-vs-last-n-days), [Fill gaps: forward fill and back fill](#fill-gaps-forward-fill-and-back-fill) |
| [Event patterns](#event-patterns) | [Sessionize events](#sessionize-events), [Find consecutive runs: gaps and islands](#find-consecutive-runs-gaps-and-islands) |
| [SQL and performance](#sql-and-performance) | [Window functions in Spark SQL](#window-functions-in-spark-sql), [How Spark runs a window, and how to keep it fast](#how-spark-runs-a-window-and-how-to-keep-it-fast) |
| [Common mistakes](#common-mistakes) | [Symptoms, causes and fixes](#symptoms-causes-and-fixes) |

## Window specifications

### `Window` and `WindowSpec`

A window specification says which rows each row is computed over. Build it from the `Window` class; every method
returns a new immutable `WindowSpec`, so specs can be shared and extended.

| method / constant | description |
| --- | --- |
| `Window.partitionBy(*cols)` | rows with equal values form one partition; omit to use the whole DataFrame (one task) |
| `Window.orderBy(*cols)` | order within each partition; accepts `F.desc(...)`, `asc_nulls_last()`, ... |
| `spec.rowsBetween(start, end)` | frame by physical row offsets relative to the current row |
| `spec.rangeBetween(start, end)` | frame by value offsets of the single numeric `orderBy` column |
| `Window.unboundedPreceding`, `Window.unboundedFollowing` | partition start and end |
| `Window.currentRow` | the current row (0) |

**Returns:** a `WindowSpec`; attach a function with `F.<function>(...).over(spec)`.

**Notes:** functions that use the same spec are computed in one pass; every different spec can add a sort and a
shuffle. Define specs once and reuse them.

In [ ]:
by_cust = Window.partitionBy("customer_id")
by_cust_time = by_cust.orderBy("order_ts", "order_id")           # extends by_cust; by_cust is unchanged
running = by_cust_time.rowsBetween(Window.unboundedPreceding, Window.currentRow)
orders.select("customer_id", "order_id", "amount",
              F.sum("amount").over(by_cust).alias("total"),
              F.row_number().over(by_cust_time).alias("n"),
              F.sum("amount").over(running).alias("running")).where("customer_id <= 2").orderBy("customer_id", "n").show()

### How a window works

A window function needs two things: the **function** (what to compute) and the **window specification** (over
which rows). You build the specification with `Window` and attach it with `.over(spec)`:

```python
spec = (
    Window.partitionBy("customer_id")  # which rows belong together
    .orderBy("order_ts")               # their order inside each partition
    .rowsBetween(-2, 0)                # optional frame: which neighbours each row sees
)
F.avg("amount").over(spec)
```

| part | question it answers | if you leave it out |
| --- | --- | --- |
| `partitionBy(cols)` | which rows are compared with each other? | the whole DataFrame is one partition |
| `orderBy(cols)` | in what order? | no order: ranking, `lag` and `lead` are not allowed |
| `rowsBetween` / `rangeBetween` | which rows around the current one? | a default frame (see [Frames](#frames-choose-which-rows-each-row-sees)) |

Three families of functions can go before `.over(...)`:

| family | functions | needs `orderBy` |
| --- | --- | --- |
| ranking | `row_number`, `rank`, `dense_rank`, `percent_rank`, `cume_dist`, `ntile` | yes |
| offset (analytic) | `lag`, `lead`, `nth_value`, `first`, `last`, `first_value`, `last_value` | yes for `lag`, `lead`, `nth_value`; recommended for the rest |
| aggregate | any aggregate: `sum`, `avg`, `count`, `min`, `max`, `collect_list`, `max_by`, ... | no |

**Use it for:**
- **Per-row context:** each order next to its customer's total, rank or previous order.
- **Replacing self-joins:** "compare with the previous row" used to need a join on `row_number - 1`.

**Notes:**
- `groupBy` returns one row per group; a window returns every input row. Choose by the shape of the output you need.
- You cannot filter on a window function in the same `where`: compute it in a column first, then filter.

## Window function catalog

Every window function in `pyspark.sql.functions`, plus aggregates used over a window. The examples use `by_time`, defined in the setup cell as `Window.partitionBy("customer_id").orderBy("order_ts", "order_id")`.

| function | what it does |
| --- | --- |
| `row_number` | 1, 2, 3,  |
| `rank`, `dense_rank` | Positions where ties share a rank: `rank` skips numbers after a tie (1, 1, 3), `dense_rank` does not (1, 1, 2) |
| `percent_rank`, `cume_dist` | Relative position: `(rank - 1) / (rows - 1)`, and the fraction of rows at or before this one |
| `ntile` | Split the ordered rows of each partition into `n` buckets as evenly as possible |
| `lag`, `lead` | The value `offset` rows before or after the current row, or `default` at the partition edge |
| `nth_value` | The value from the nth row of the frame (1-based), or null if the frame is shorter |
| `first`, `last` over a window | The first or last value in the frame; `ignorenulls=True` skips nulls, which gives forward fill |
| Aggregates over a window | Any aggregate with `.over(spec)`: a group value on every row, or a running value with a frame |

### `row_number`

1, 2, 3, ... in window order, with no ties.

**Signature:** `F.row_number().over(spec)`

| parameter | type | description |
| --- | --- | --- |
| `(none)` |  | attach with `.over(spec)`; the spec needs an `orderBy` |

**Use it for:** Deduplication (keep `rn = 1`) and exactly-N-per-group results.

**Notes:** Tied rows are numbered in an arbitrary order; add a tie-breaker to `orderBy`.

In [ ]:
orders.select("customer_id", "order_id", "order_ts", F.row_number().over(by_time).alias("row_number")).where("customer_id <= 2").show()

### `rank`, `dense_rank`

Positions where ties share a rank: `rank` skips numbers after a tie (1, 1, 3), `dense_rank` does not (1, 1, 2).

**Signature:** `F.rank().over(spec)` · `F.dense_rank().over(spec)`

**Use it for:** Leaderboards and top-N where ties should all be kept.

**Notes:** Pick `dense_rank` for "top N values" and `rank` for "top N places".

In [ ]:
scores = spark.createDataFrame([("Ana", 300), ("Ben", 300), ("Cai", 200), ("Dee", 100)], "rep string, revenue int")
by_revenue = Window.orderBy(F.desc("revenue"))
scores.select("rep", "revenue", F.rank().over(by_revenue).alias("rank"), F.dense_rank().over(by_revenue).alias("dense_rank")).show()

### `percent_rank`, `cume_dist`

Relative position: `(rank - 1) / (rows - 1)`, and the fraction of rows at or before this one.

**Signature:** `F.percent_rank().over(spec)` · `F.cume_dist().over(spec)`

**Use it for:** Percentile positions and top-10-percent filters.

**Notes:** Tied rows share the same value.

In [ ]:
scores.select("rep", "revenue", F.round(F.percent_rank().over(by_revenue), 2).alias("percent_rank"),
              F.round(F.cume_dist().over(by_revenue), 2).alias("cume_dist")).show()

### `ntile`

Split the ordered rows of each partition into `n` buckets as evenly as possible.

**Signature:** `F.ntile(n).over(spec)`

| parameter | type | description |
| --- | --- | --- |
| `n` | int | number of buckets |

**Use it for:** Quartiles and deciles of customers by spend.

**Notes:** When rows do not divide evenly, the first buckets get the extra rows.

In [ ]:
orders.select("order_id", "amount", F.ntile(4).over(Window.orderBy("amount")).alias("quartile")).orderBy("amount").show(6)

### `lag`, `lead`

The value `offset` rows before or after the current row, or `default` at the partition edge.

**Signature:** `F.lag(col, offset=1, default=None).over(spec)` · `F.lead(col, offset=1, default=None).over(spec)`

| parameter | type | description |
| --- | --- | --- |
| `col` | Column or name | value to read |
| `offset` | int, default 1 | rows before (`lag`) or after (`lead`) |
| `default` | any, default None | returned when the offset row does not exist |

**Use it for:** Time since the previous event, change from the previous value, the next status.

**Notes:** Without `partitionBy`, `lag` reads across entities at the boundaries.

In [ ]:
orders.select("customer_id", "order_id", "amount", F.lag("amount").over(by_time).alias("prev_amount"),
              F.lead("order_id", 1, -1).over(by_time).alias("next_order")).where("customer_id <= 2").show()

### `nth_value`

The value from the nth row of the frame (1-based), or null if the frame is shorter.

**Signature:** `F.nth_value(col, n, ignoreNulls=False).over(spec)`

| parameter | type | description |
| --- | --- | --- |
| `col` | Column or name | value to read |
| `offset` | int | 1-based row within the frame |
| `ignoreNulls` | bool, default False | skip nulls when counting |

**Use it for:** Each customer's second order, a common retention metric.

**Notes:** With the default frame, rows before the nth get null; use a whole-partition frame to fill every row.

In [ ]:
whole = by_time.rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)
orders.select("customer_id", "order_id", F.nth_value("order_id", 2).over(by_time).alias("default_frame"),
              F.nth_value("order_id", 2).over(whole).alias("whole_partition")).where("customer_id <= 2").show()

### `first`, `last` over a window

The first or last value in the frame; `ignorenulls=True` skips nulls, which gives forward fill.

**Signature:** `F.first(col, ignorenulls).over(spec)` · `F.last(col, ignorenulls).over(spec)`

| parameter | type | description |
| --- | --- | --- |
| `col` | Column or name | value to read |
| `ignorenulls` | bool, default False | skip nulls (forward / back fill) |

**Use it for:** First and latest values per entity, and filling gaps in sparse readings.

**Notes:** With `orderBy` and no frame, `last` returns the current row; set `rowsBetween` explicitly.

In [ ]:
orders.select("customer_id", "order_id", F.first("order_id").over(by_time).alias("first"), F.last("order_id").over(by_time).alias("last_default"),
              F.last("order_id").over(whole).alias("last_whole")).where("customer_id <= 2").show()

### Aggregates over a window

Any aggregate with `.over(spec)`: a group value on every row, or a running value with a frame.

**Signature:** `F.sum(col).over(spec)`, `F.avg(col).over(spec.rowsBetween(-2, 0))`, ...

**Use it for:** Shares of a total, running totals and moving averages.

**Notes:** Adding `orderBy` changes the default frame, turning a group total into a running total.

In [ ]:
orders.select("customer_id", "order_id", "amount", F.sum("amount").over(Window.partitionBy("customer_id")).alias("customer_total"),
              F.sum("amount").over(by_time.rowsBetween(Window.unboundedPreceding, 0)).alias("running_total")).where("customer_id <= 2").show()

In [ ]:
orders.groupBy("customer_id").agg(F.sum("amount").alias("customer_total")).orderBy("customer_id").show(3)

by_customer = Window.partitionBy("customer_id")
orders.select(
    "customer_id", "order_id", "amount",
    F.sum("amount").over(by_customer).alias("customer_total"),
).orderBy("customer_id", "order_id").show(5)

## Aggregates and ranking

### Aggregates over a whole partition

Any aggregate function becomes a window function with `.over(spec)`. With only `partitionBy`, every row sees its
whole partition, so the result is the same on each row of a partition: the group total, average, maximum or count.
Typical uses are **shares** (row / total), **differences from the group** (row - average) and **filters against
the group** (rows above their group's average).

**Use it for:**
- **Contribution analysis:** each product's share of its category's revenue.
- **Outlier checks:** orders more than three times their customer's average.

**Notes:**
- Add `orderBy` and the aggregate silently becomes a running aggregate, because the default frame changes. Keep
  group-level aggregates on a specification with no order.

In [ ]:
with_context = orders.select(
    "customer_id", "order_id", "amount",
    F.sum("amount").over(by_customer).alias("total"),
    F.round(F.avg("amount").over(by_customer), 2).alias("avg"),
    F.max("amount").over(by_customer).alias("largest"),
    F.count("*").over(by_customer).alias("orders"),
    F.round(F.col("amount") / F.sum("amount").over(by_customer) * 100, 1).alias("pct_of_customer"),
    (F.col("amount") - F.avg("amount").over(by_customer)).cast("decimal(12,2)").alias("vs_avg"),
)
with_context.orderBy("customer_id", "order_id").show(6)
with_context.where("amount > avg").select("customer_id", "order_id", "amount", "avg").orderBy("customer_id").show()

### Rank rows: row_number, rank and dense_rank

All three number rows within each partition in the window's order. They differ only on **ties**:

| function | values for 300, 300, 200, 100 | use when |
| --- | --- | --- |
| `row_number()` | 1, 2, 3, 4 | you need exactly one row per position (dedup, top-1) |
| `rank()` | 1, 1, 3, 4 | ties share a place and the next place is skipped, as in sports |
| `dense_rank()` | 1, 1, 2, 3 | ties share a place and no number is skipped ("top 2 values") |

**Use it for:**
- **Leaderboards** per region, store or category.
- **Exactly one row per key** with `row_number`.

**Notes:**
- With ties, `row_number` picks an order you cannot rely on: two runs can number the tied rows differently. Add
  a unique column to the `orderBy` as a tie-breaker.
- Nulls sort first in ascending order; use `F.desc_nulls_last` or `F.asc_nulls_last` when they should rank last.

In [ ]:
sales = spark.createDataFrame(
    [("north", "Ana", 300), ("north", "Ben", 300), ("north", "Cai", 200), ("north", "Dee", 100),
     ("south", "Eli", 500), ("south", "Fay", None)],
    "region string, rep string, revenue int",
)
by_revenue = Window.partitionBy("region").orderBy(F.desc_nulls_last("revenue"))
by_revenue_stable = Window.partitionBy("region").orderBy(F.desc_nulls_last("revenue"), "rep")
sales.select(
    "region", "rep", "revenue",
    F.row_number().over(by_revenue).alias("row_number"),
    F.row_number().over(by_revenue_stable).alias("row_number_stable"),
    F.rank().over(by_revenue).alias("rank"),
    F.dense_rank().over(by_revenue).alias("dense_rank"),
).orderBy("region", "row_number_stable").show()

### Relative position: percent_rank, cume_dist and ntile

| function | formula | values for 4 rows |
| --- | --- | --- |
| `percent_rank()` | (rank - 1) / (rows - 1) | 0, 0.33, 0.67, 1 |
| `cume_dist()` | rows at or before this one / rows | 0.25, 0.5, 0.75, 1 |
| `ntile(n)` | splits the ordered rows into `n` buckets as evenly as possible | `ntile(2)`: 1, 1, 2, 2 |

**Use it for:**
- **Segmentation:** quartiles of customers by spend (`ntile(4)`), the top 10 percent (`percent_rank`).
- **Percentiles per group** without collecting the data.

**Notes:**
- When rows do not divide evenly, `ntile` gives the extra rows to the first buckets.
- Ties share a `percent_rank` and a `cume_dist`, but `ntile` may put tied rows in different buckets.

In [ ]:
by_amount_all = Window.orderBy("amount")  # a single partition: fine for 12 rows, see the performance section
orders.select(
    "order_id", "amount",
    F.ntile(4).over(by_amount_all).alias("quartile"),
    F.round(F.percent_rank().over(by_amount_all), 2).alias("percent_rank"),
    F.round(F.cume_dist().over(by_amount_all), 2).alias("cume_dist"),
).orderBy("amount").show()

### Keep the top N rows per group

Rank inside each partition, then filter on the rank. The ranking function decides what "top 2" means when there
are ties: `row_number` returns exactly two rows, `rank` and `dense_rank` return every row tied for a place.

**Use it for:**
- **Best sellers per category,** largest orders per customer, highest-paid employee per department.

**Notes:**
- Filter in a separate step after the window column exists.
- Decide with the business whether ties should all be kept; it changes how many rows a report has.

In [ ]:
ranked = sales.select(
    "region", "rep", "revenue",
    F.row_number().over(by_revenue_stable).alias("rn"),
    F.rank().over(by_revenue).alias("rnk"),
    F.dense_rank().over(by_revenue).alias("drnk"),
)
print("row_number <= 2:", [r.rep for r in ranked.where("rn <= 2").orderBy("region", "rn").collect()])
print("rank <= 2:      ", [r.rep for r in ranked.where("rnk <= 2").orderBy("region", "rnk", "rep").collect()])
print("dense_rank <= 2:", [r.rep for r in ranked.where("drnk <= 2").orderBy("region", "drnk", "rep").collect()])

## Deduplication and offsets

### Deduplicate: keep the latest record per key

Change feeds, retries and replays deliver several versions of one key. Number each key's versions with
`row_number` ordered newest first, by the **source** sequence (log sequence number, version, the source's
`updated_at`) plus a stable tie-breaker, and keep `rn = 1`.

`dropDuplicates(["key"])` also keeps one row per key, but which row it keeps is not defined.

**Use it for:**
- **Silver tables** built from CDC: one current row per order.
- **Reprocessing:** a replayed batch must pick the same winners as the first run.

**Notes:**
- Ingestion time changes on a replay; use it only as the last tie-breaker.
- `max_by(status, lsn)` in a `groupBy` gives the latest value of one column; the window keeps the whole row.

In [ ]:
changes = spark.createDataFrame(
    [(101, "placed", 1, datetime(2026, 1, 5, 9, 0)), (101, "paid", 2, datetime(2026, 1, 5, 9, 5)),
     (101, "shipped", 3, datetime(2026, 1, 6, 8, 0)), (102, "placed", 1, datetime(2026, 1, 5, 10, 0)),
     (102, "cancelled", 2, datetime(2026, 1, 5, 10, 1)), (102, "cancelled", 2, datetime(2026, 1, 5, 10, 9))],
    "order_id int, status string, source_lsn long, ingested_at timestamp",
)
newest_first = Window.partitionBy("order_id").orderBy(F.desc("source_lsn"), F.desc("ingested_at"))
latest = changes.withColumn("rn", F.row_number().over(newest_first)).where("rn = 1").drop("rn")
latest.orderBy("order_id").show()
changes.groupBy("order_id").agg(F.max_by("status", "source_lsn").alias("latest_status")).orderBy("order_id").show()

### Look back and ahead: lag and lead

`F.lag(col, offset=1, default=None)` reads the value `offset` rows **before** the current row in window order;
`F.lead(col, offset=1, default=None)` reads it `offset` rows **after**. At the edge of a partition there is no such
row, so they return `default` (null unless you pass one).

**Use it for:**
- **Gaps:** days since the previous order, minutes between events.
- **Change:** difference or percentage change from the previous value; the next status after this one.

**Notes:**
- Without `partitionBy`, `lag` reads the previous *customer's* row at each boundary.
- Ties in the `orderBy` make "previous" ambiguous; add a tie-breaker.
- A `default` of 0 can hide that there was no previous row. Use it only when that is truly what 0 means.

In [ ]:
in_order = Window.partitionBy("customer_id").orderBy("order_ts", "order_id")
orders.where("customer_id IN (1, 2, 3)").select(
    "customer_id", "order_id", "order_ts", "amount",
    F.lag("amount").over(in_order).alias("prev_amount"),
    (F.col("amount") - F.lag("amount").over(in_order)).alias("change"),
    F.round((F.unix_timestamp("order_ts") - F.unix_timestamp(F.lag("order_ts").over(in_order))) / 3600, 1)
     .alias("hours_since_prev"),
    F.lead("order_id").over(in_order).alias("next_order"),
    F.lead("order_id", 2, -1).over(in_order).alias("order_after_next"),
).orderBy("customer_id", "order_ts").show()

### Detect changes and number the runs

A common use of `lag` is a **change flag**: 1 when a value differs from the previous row, else 0. A running sum
of the flags then gives each unbroken run of the same value its own number, so you can group runs.

In [ ]:
status_log = spark.createDataFrame(
    [("M1", 1, "up"), ("M1", 2, "up"), ("M1", 3, "down"), ("M1", 4, "down"), ("M1", 5, "up"),
     ("M2", 1, "down"), ("M2", 2, "up")],
    "machine string, minute int, state string",
)
by_minute = Window.partitionBy("machine").orderBy("minute")
runs = (
    status_log
    .withColumn("changed", F.when(F.lag("state").over(by_minute).eqNullSafe(F.col("state")), 0).otherwise(1))
    .withColumn("run_id", F.sum("changed").over(by_minute.rowsBetween(Window.unboundedPreceding, 0)))
)
runs.orderBy("machine", "minute").show()
runs.groupBy("machine", "run_id", "state").agg(
    F.min("minute").alias("from_minute"), F.max("minute").alias("to_minute")
).orderBy("machine", "run_id").show()

### first, last and nth_value

| function | returns |
| --- | --- |
| `first(col, ignorenulls=False)`, `first_value` | the first value in the frame |
| `last(col, ignorenulls=False)`, `last_value` | the last value in the frame |
| `nth_value(col, n, ignoreNulls=False)` | the value in the `n`th row of the frame (1-based), or null if the frame is shorter |

All three read the **frame**, not the whole partition. With an `orderBy` and no explicit frame, the frame ends at
the current row, so `last` returns the current row's value and `nth_value(col, 2)` is null on the first row. Use
`rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)` to see the whole partition.

**Use it for:**
- **First and latest values:** first order amount, current status, the second purchase (a classic retention metric).

**Notes:**
- `ignorenulls=True` skips nulls, which is what forward fill uses; without it a null in the frame is returned as is.

In [ ]:
whole = in_order.rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)
orders.where("customer_id IN (1, 2)").select(
    "customer_id", "order_id",
    F.first("order_id").over(in_order).alias("first_default"),
    F.last("order_id").over(in_order).alias("last_default"),
    F.last("order_id").over(whole).alias("last_whole"),
    F.nth_value("order_id", 2).over(in_order).alias("second_default"),
    F.nth_value("order_id", 2).over(whole).alias("second_whole"),
).orderBy("customer_id", "order_id").show()

## Frames and running metrics

### Frames: choose which rows each row sees

A frame limits the rows of the partition that each row's function reads, relative to the current row.

| frame | meaning |
| --- | --- |
| `rowsBetween(start, end)` | physical rows: `-2, 0` is the two previous rows and this one |
| `rangeBetween(start, end)` | values of the single `orderBy` column: `-7, 0` is every row whose value is within 7 below |
| `Window.unboundedPreceding`, `Window.unboundedFollowing`, `Window.currentRow` | the partition start, the partition end, this row (0) |

The default frame depends on whether the window has an order:

| specification | default frame |
| --- | --- |
| `partitionBy` only | the whole partition |
| `partitionBy` + `orderBy` | `RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW` |

That `RANGE` default includes every row that **ties** with the current row in the ordering, which surprises people
writing running totals.

**Use it for:**
- **Running totals** (start to current row) and **moving windows** (a fixed number of neighbours).
- **Calendar windows** such as "the last 7 days", which need `rangeBetween` over a number.

**Notes:**
- Ranking functions and `lag` / `lead` ignore frames.
- `rangeBetween` needs exactly one numeric (or date-as-number) `orderBy` column.

In [ ]:
payments = spark.createDataFrame(
    [(1, "09:00", 10), (2, "09:05", 20), (3, "09:05", 30), (4, "09:10", 40)],
    "payment_id int, at string, amount int",
)
by_time = Window.orderBy("at")
payments.select(
    "payment_id", "at", "amount",
    F.sum("amount").over(by_time).alias("default_range"),
    F.sum("amount").over(by_time.rowsBetween(Window.unboundedPreceding, Window.currentRow)).alias("rows_frame"),
    F.sum("amount").over(Window.orderBy("at", "payment_id").rowsBetween(Window.unboundedPreceding, 0))
     .alias("rows_with_tiebreak"),
).orderBy("payment_id").show()

### Running totals and cumulative metrics

A frame from `Window.unboundedPreceding` to `Window.currentRow` gives everything so far: running sum, running
count, running maximum, or a running average. Divide a running sum by the partition total for a cumulative share.

**Use it for:**
- **Cumulative revenue** per customer or per month, year-to-date figures.
- **Pareto analysis:** how many customers make up 80 percent of revenue.

**Notes:**
- Order by something unique, or ties share a running value (the default `RANGE` frame) or are numbered arbitrarily.

In [ ]:
so_far = in_order.rowsBetween(Window.unboundedPreceding, Window.currentRow)
orders.where("customer_id IN (1, 3, 4)").select(
    "customer_id", "order_id", "amount",
    F.sum("amount").over(so_far).alias("running_total"),
    F.count("*").over(so_far).alias("running_orders"),
    F.max("amount").over(so_far).alias("running_max"),
    F.round(F.sum("amount").over(so_far) / F.sum("amount").over(by_customer) * 100, 1).alias("cum_pct"),
).orderBy("customer_id", "order_id").show()

### Moving windows: last N rows vs last N days

- `rowsBetween(-2, 0)`: this row and the two before it, a **3-row** moving window, whatever their dates.
- `rangeBetween(-2, 0)` over a day number: every row from the **last 3 calendar days**, however many rows.
- `rowsBetween(-1, 1)`: a centred window, smoothing each point with its neighbours.

To use `rangeBetween` with dates, order by a number: `F.unix_date("day")` (days since 1970-01-01) for dates, or
`F.col("ts").cast("long")` (seconds) for timestamps.

**Use it for:**
- **Smoothing dashboards** with a 7-day moving average.
- **Fraud and rate limits:** spend or attempts in the last 24 hours.

**Notes:**
- A rows frame over data with missing days silently stretches across the gap; a range frame does not.
- The first rows of each partition average fewer values; filter them out if a full window is required.

In [ ]:
daily_sales = spark.createDataFrame(
    [(date(2026, 1, 1), 100), (date(2026, 1, 2), 200), (date(2026, 1, 3), 300),
     (date(2026, 1, 5), 500), (date(2026, 1, 6), 600)],  # no row for 4 January
    "day date, revenue int",
)
by_day_rows = Window.orderBy("day")
by_day_range = Window.orderBy(F.unix_date("day"))
daily_sales.select(
    "day", "revenue",
    F.avg("revenue").over(by_day_rows.rowsBetween(-2, 0)).alias("avg_3_rows"),
    F.avg("revenue").over(by_day_range.rangeBetween(-2, 0)).alias("avg_3_days"),
    F.count("*").over(by_day_range.rangeBetween(-2, 0)).alias("days_with_data"),
    F.avg("revenue").over(by_day_rows.rowsBetween(-1, 1)).alias("centred_3"),
).orderBy("day").show()

### Fill gaps: forward fill and back fill

Sensors and slowly changing attributes often send a value only when it changes, leaving nulls in between.

- **Forward fill** (carry the last known value down): `F.last(col, ignorenulls=True)` over
  `rowsBetween(Window.unboundedPreceding, Window.currentRow)`.
- **Back fill** (pull the next known value up): `F.first(col, ignorenulls=True)` over
  `rowsBetween(Window.currentRow, Window.unboundedFollowing)`.

**Use it for:**
- **IoT readings** and price feeds sent on change only.
- **Attributes** such as a customer's tier between updates.

**Notes:**
- Forward fill cannot fill nulls before the first known value; decide whether to back fill those or leave them.
- Fill only when "unchanged since the last reading" is really what a null means.

In [ ]:
readings = spark.createDataFrame(
    [("S1", 1, None), ("S1", 2, 20.5), ("S1", 3, None), ("S1", 4, None), ("S1", 5, 22.0),
     ("S2", 1, 18.0), ("S2", 2, None)],
    "sensor string, minute int, temp double",
)
by_sensor = Window.partitionBy("sensor").orderBy("minute")
readings.select(
    "sensor", "minute", "temp",
    F.last("temp", ignorenulls=True).over(by_sensor.rowsBetween(Window.unboundedPreceding, 0)).alias("forward_fill"),
    F.first("temp", ignorenulls=True).over(by_sensor.rowsBetween(0, Window.unboundedFollowing)).alias("back_fill"),
).orderBy("sensor", "minute").show()

## Event patterns

### Sessionize events

A **session** is a burst of activity separated by inactivity, for example more than 30 minutes without an event.
The recipe combines three window steps:

1. `lag` gives the gap since the user's previous event;
2. `when` flags a new session when the gap is over the timeout or there is no previous event;
3. a running `sum` of the flags numbers the sessions.

Structured Streaming offers `session_window` for the same idea on streams
([lesson 11](pyspark-11-structured-streaming.ipynb)).

**Use it for:**
- **Web and app analytics:** sessions per user, session length, conversion per session.

**Notes:**
- Partition by user, or sessions leak across users.
- Deduplicate retried events first, or a retry looks like activity.

In [ ]:
clicks = spark.createDataFrame(
    [("u1", datetime(2026, 1, 6, 9, 0)), ("u1", datetime(2026, 1, 6, 9, 10)), ("u1", datetime(2026, 1, 6, 9, 25)),
     ("u1", datetime(2026, 1, 6, 11, 0)), ("u2", datetime(2026, 1, 6, 9, 0)), ("u2", datetime(2026, 1, 6, 9, 45))],
    "user_id string, ts timestamp",
)
user_time = Window.partitionBy("user_id").orderBy("ts")
sessions = (
    clicks
    .withColumn("gap_min", (F.col("ts").cast("long") - F.lag("ts").over(user_time).cast("long")) / 60)
    .withColumn("new_session", F.when(F.col("gap_min").isNull() | (F.col("gap_min") > 30), 1).otherwise(0))
    .withColumn("session", F.sum("new_session").over(user_time.rowsBetween(Window.unboundedPreceding, 0)))
)
sessions.orderBy("user_id", "ts").show()
sessions.groupBy("user_id", "session").agg(
    F.count("*").alias("clicks"),
    F.min("ts").alias("started"),
    ((F.max("ts").cast("long") - F.min("ts").cast("long")) / 60).alias("minutes"),
).orderBy("user_id", "session").show()

### Find consecutive runs: gaps and islands

"Longest streak of consecutive active days" is a **gaps-and-islands** problem. Number each user's distinct days
with `row_number` and subtract that number from the date. Inside an unbroken run, the date and the row number both
go up by one each step, so the difference stays the same; grouping by it finds each run (an island).

**Use it for:**
- **Engagement:** login streaks, consecutive months with a purchase.
- **Operations:** periods of continuous downtime.

**Notes:**
- Deduplicate to one row per user and day first, or a repeated day breaks the arithmetic.

In [ ]:
logins = spark.createDataFrame(
    [("u1", "2026-01-01"), ("u1", "2026-01-02"), ("u1", "2026-01-02"), ("u1", "2026-01-03"),
     ("u1", "2026-01-05"), ("u1", "2026-01-06"), ("u2", "2026-01-10"), ("u3", "2026-01-01"), ("u3", "2026-01-03")],
    "user_id string, login_date string",
)
days = logins.select("user_id", F.to_date("login_date").alias("day")).distinct()
islands = days.withColumn("island", F.date_sub("day", F.row_number().over(Window.partitionBy("user_id").orderBy("day"))))
streaks = islands.groupBy("user_id", "island").agg(
    F.count("*").alias("days"), F.min("day").alias("start"), F.max("day").alias("end")
)
longest = Window.partitionBy("user_id").orderBy(F.desc("days"), "start")
streaks.withColumn("rn", F.row_number().over(longest)).where("rn = 1").drop("rn", "island").orderBy("user_id").show()

## SQL and performance

### Window functions in Spark SQL

The same windows in SQL use `OVER (PARTITION BY ... ORDER BY ... frame)`. A named `WINDOW` clause lets several
functions share one specification. Spark 4.0 has no `QUALIFY` clause, so filter on a window result in an outer
query or a CTE.

```sql
SELECT customer_id, order_id, amount,
       row_number() OVER w                                   AS rn,
       sum(amount)  OVER (w ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS running_total
FROM orders
WINDOW w AS (PARTITION BY customer_id ORDER BY order_ts, order_id)
```

**Use it for:**
- **Analysts' queries** and dbt models that move into a Spark job.

**Notes:**
- The same frame defaults apply in SQL: `ORDER BY` without a frame means `RANGE ... CURRENT ROW`.

In [ ]:
spark.sql('''
    WITH ranked AS (
        SELECT customer_id, order_id, amount,
               row_number() OVER w AS rn,
               sum(amount) OVER (PARTITION BY customer_id) AS customer_total
        FROM {orders}
        WINDOW w AS (PARTITION BY customer_id ORDER BY amount DESC, order_id)
    )
    SELECT customer_id, order_id, amount, customer_total
    FROM ranked
    WHERE rn = 1
    ORDER BY customer_id
''', orders=orders).show()

### How Spark runs a window, and how to keep it fast

For each window specification Spark **shuffles** rows by the `partitionBy` keys (an `Exchange`), **sorts** each
partition by the partition and order keys, then evaluates all functions that share that specification in one
`Window` operator. So:

- Functions over the **same** specification are computed together, in one shuffle and sort.
- Each **different** specification can add another sort, and another shuffle if its partition keys differ.
- A window **without `partitionBy`** moves every row to one partition (`Exchange SinglePartition`) and one task:
  fine for a small summary, a failure on a large table.
- One huge partition key (a "bot" user with millions of events) makes one task much slower than the rest.

**Use it for:**
- **Large fact tables,** where every extra shuffle costs minutes.

**Notes:**
- Reuse a specification object for related columns rather than writing near-identical ones.
- Aggregate first when you only need one row per group; a window over billions of rows is more expensive.

In [ ]:
import contextlib
import io


def plan_summary(df):
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        df.explain()
    lines = [line.strip() for line in buffer.getvalue().splitlines()]
    windows = sum(line.lstrip("+- ").startswith("Window ") for line in lines)
    exchanges = [line.split("Exchange ")[1].split(",")[0] for line in lines if "Exchange " in line]
    return f"Window operators: {windows}, shuffles: {exchanges}"


same_spec = orders.select(
    "*", F.row_number().over(in_order).alias("rn"), F.lag("amount").over(in_order).alias("prev"),
    F.sum("amount").over(in_order.rowsBetween(Window.unboundedPreceding, 0)).alias("running"),
)
two_specs = orders.select(
    "*", F.row_number().over(in_order).alias("rn"),
    F.rank().over(Window.partitionBy("status").orderBy("amount")).alias("status_rank"),
)
no_partition = orders.select("*", F.row_number().over(Window.orderBy("amount")).alias("overall_rank"))

print("same spec:   ", plan_summary(same_spec))
print("two specs:   ", plan_summary(two_specs))
print("no partition:", plan_summary(no_partition))

## Common mistakes

### Symptoms, causes and fixes

| symptom | cause | fix |
| --- | --- | --- |
| `last` returns the current row | default frame ends at the current row | `rowsBetween(unboundedPreceding, unboundedFollowing)` |
| a "group total" grows row by row | `orderBy` added to an aggregate's window | use a specification without `orderBy` |
| tied rows share a running total | default `RANGE` frame | `rowsBetween(...)` and a tie-breaker |
| results change between runs | `row_number` over non-unique ordering | add a unique column to `orderBy` |
| `lag` mixes customers | no `partitionBy` | partition by the entity |
| one task runs for hours | no `partitionBy`, or one huge key | partition the window; handle skewed keys separately |
| `AnalysisException` filtering on a window | window used inside `where` | compute it into a column, then filter |

**Next:** practise these entries in [PySpark practice 8: window functions](pyspark-practice-08-window-functions.ipynb).